In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [8]:
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [9]:
import zipfile

with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('/content/caltech101')

print("Extracted!")

Extracted!


In [10]:
data_dir = "/content/caltech101/caltech-101"

In [11]:
import os
print(os.listdir(data_dir)[:10])

['strawberry', 'Faces', 'wrench', 'flamingo_head', 'hedgehog', 'dollar_bill', 'gerenuk', 'okapi', 'soccer_ball', 'cellphone']


In [12]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

data_dir = "/content/caltech101/caltech-101"

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print("Total images:", len(dataset))
print("Classes:", len(dataset.classes))
print(dataset.classes[:5])


Total images: 9144
Classes: 102
['BACKGROUND_Google', 'Faces', 'Faces_easy', 'Leopards', 'Motorbikes']


In [13]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 7315
Test size: 1829


In [14]:
import torch
import torch.nn as nn

class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self, x):
        x = self.features(x)
        return x.view(x.size(0), -1)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNFeatureExtractor().to(device)
model.eval()

CNNFeatureExtractor(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): AdaptiveAvgPool2d(output_size=(1, 1))
  )
)

In [16]:
import numpy as np

def extract_features(loader, model):
    features = []
    labels = []

    with torch.no_grad():
        for images, lbls in loader:
            images = images.to(device)
            outputs = model(images)
            features.append(outputs.cpu().numpy())
            labels.extend(lbls.numpy())

    return np.vstack(features), np.array(labels)

train_features, train_labels = extract_features(train_loader, model)
test_features, test_labels = extract_features(test_loader, model)

print("Feature shape:", train_features.shape)

Feature shape: (7315, 256)


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query_feature, database_features, k=5):
    sim = cosine_similarity(query_feature.reshape(1, -1), database_features)
    indices = np.argsort(sim[0])[::-1][:k]
    return indices

In [18]:
import matplotlib.pyplot as plt

def show_results(query_idx, k=5):
    query_feature = test_features[query_idx]
    indices = retrieve(query_feature, train_features, k)

    plt.figure(figsize=(12,3))

    img, _ = test_dataset[query_idx]
    plt.subplot(1, k+1, 1)
    plt.imshow(img.permute(1,2,0))
    plt.title("Query")
    plt.axis('off')

    for i, idx in enumerate(indices):
        img, _ = train_dataset[idx]
        plt.subplot(1, k+1, i+2)
        plt.imshow(img.permute(1,2,0))
        plt.axis('off')

    plt.show()

In [19]:
def evaluate(test_features, test_labels, train_features, train_labels, k=5):
    precision_list = []
    recall_list = []

    for i in range(len(test_features)):
        query = test_features[i]
        true_label = test_labels[i]

        indices = retrieve(query, train_features, k)
        retrieved_labels = train_labels[indices]

        relevant = np.sum(retrieved_labels == true_label)

        precision = relevant / k
        recall = relevant / np.sum(train_labels == true_label)

        precision_list.append(precision)
        recall_list.append(recall)

    return np.mean(precision_list), np.mean(recall_list)

precision, recall = evaluate(test_features, test_labels, train_features, train_labels)

print("Precision:", precision)
print("Recall:", recall)

Precision: 0.25784581738655005
Recall: 0.007917834846135162


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query_feature, database_features, k=5):
    sim = cosine_similarity(query_feature.reshape(1, -1), database_features)
    indices = np.argsort(sim[0])[::-1][:k]
    return indices

In [21]:
def evaluate_cnn(test_features, test_labels, train_features, train_labels, k=5):
    precision_list = []
    recall_list = []

    for i in range(len(test_features)):
        query = test_features[i]
        true_label = test_labels[i]

        indices = retrieve(query, train_features, k)
        retrieved_labels = train_labels[indices]

        relevant = np.sum(retrieved_labels == true_label)

        precision = relevant / k
        recall = relevant / np.sum(train_labels == true_label)

        precision_list.append(precision)
        recall_list.append(recall)

    return np.mean(precision_list), np.mean(recall_list)

In [22]:
def compute_map_cnn(test_features, test_labels, train_features, train_labels, k=5):
    APs = []

    for i in range(len(test_features)):
        query = test_features[i]
        true_label = test_labels[i]

        indices = retrieve(query, train_features, k)
        retrieved_labels = train_labels[indices]

        correct = 0
        precision_sum = 0

        for j in range(k):
            if retrieved_labels[j] == true_label:
                correct += 1
                precision_sum += correct / (j+1)

        AP = precision_sum / max(correct, 1)
        APs.append(AP)

    return np.mean(APs)

In [23]:
precision, recall = evaluate_cnn(
    test_features, test_labels,
    train_features, train_labels,
    k=5
)

mAP = compute_map_cnn(
    test_features, test_labels,
    train_features, train_labels,
    k=5
)

print("CNN Results:")
print("Precision:", precision)
print("Recall:", recall)
print("mAP:", mAP)

CNN Results:
Precision: 0.25784581738655005
Recall: 0.007917834846135162
mAP: 0.35716238381629306


In [25]:
class CaltechCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))

        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self, x):
        f1 = self.conv_block1(x)
        f2 = self.conv_block2(f1)
        f3 = self.conv_block3(f2)
        f4 = self.conv_block4(f3)

        emb = f4.view(f4.size(0), -1)
        return f4, emb   # IMPORTANT

In [26]:
def extract_embeddings(model, dataset, N):
    feats, labels, imgs = [], [], []

    model.eval()
    with torch.no_grad():
        for i in range(N):
            img, lbl = dataset[i]
            inp = img.unsqueeze(0).to(device)

            _, emb = model(inp)

            feats.append(emb.cpu().numpy()[0])
            labels.append(lbl)
            imgs.append(img)

    return np.array(feats), np.array(labels), np.array(imgs)

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query_feat, database_feats, k=5):
    sims = cosine_similarity(query_feat.reshape(1, -1), database_feats)[0]
    idx = np.argsort(sims)[::-1][:k]
    return idx, sims[idx]

In [28]:
def precision_at_k(retrieved_labels, true_label, k):
    return np.sum(retrieved_labels[:k] == true_label) / k

In [29]:
def evaluate(feats, labels, k=5):
    precisions, recalls, APs = [], [], []

    for i in range(len(feats)):
        mask = np.ones(len(feats), dtype=bool)
        mask[i] = False

        query_feat = feats[i]
        true_label = labels[i]

        idx, _ = retrieve(query_feat, feats[mask], k)
        ret_labels = labels[mask][idx]

        # Precision
        prec = precision_at_k(ret_labels, true_label, k)
        precisions.append(prec)

        # Recall
        total_relevant = np.sum(labels == true_label) - 1
        rec = np.sum(ret_labels == true_label) / total_relevant
        recalls.append(rec)

        # mAP
        correct = 0
        precision_sum = 0
        for j in range(k):
            if ret_labels[j] == true_label:
                correct += 1
                precision_sum += correct / (j+1)

        AP = precision_sum / max(correct, 1)
        APs.append(AP)

    return {
        f'Precision@{k}': np.mean(precisions),
        f'Recall@{k}': np.mean(recalls),
        'mAP': np.mean(APs)
    }

In [30]:
import torchvision.transforms.functional as TF

def apply_distortion(img, dtype):
    if dtype == 'noise':
        img = img + torch.randn_like(img) * 0.2
        img = torch.clamp(img, 0, 1)
    elif dtype == 'blur':
        img = TF.gaussian_blur(img, 5)
    elif dtype == 'rotation':
        img = TF.rotate(img, 15)
    return img